In [1]:
import pandas as pd

In [5]:
df = pd.read_csv(r"C:\Users\Admin\OneDrive\Documents\ecommerce_sales.csv")


In [7]:
print(df.head(2))

  order_id customer_id product_id category   price  discount  quantity  \
0  O100000      C17270    P234890     Home  164.08      0.15         1   
1  O100001      C17603    P228204  Grocery   24.73      0.00         1   

  payment_method  order_date  delivery_time_days region returned  \
0    Credit Card  23-12-2023                   4   West       No   
1    Credit Card  03-04-2025                   6  South       No   

   total_amount  shipping_cost  profit_margin  customer_age customer_gender  
0        139.47           7.88          31.17            60          Female  
1         24.73           4.60          -2.62            37            Male  


In [8]:
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 34500 entries, 0 to 34499
Data columns (total 17 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   order_id            34500 non-null  str    
 1   customer_id         34500 non-null  str    
 2   product_id          34500 non-null  str    
 3   category            34500 non-null  str    
 4   price               34500 non-null  float64
 5   discount            34500 non-null  float64
 6   quantity            34500 non-null  int64  
 7   payment_method      34500 non-null  str    
 8   order_date          34500 non-null  str    
 9   delivery_time_days  34500 non-null  int64  
 10  region              34500 non-null  str    
 11  returned            34500 non-null  str    
 12  total_amount        34500 non-null  float64
 13  shipping_cost       34500 non-null  float64
 14  profit_margin       34500 non-null  float64
 15  customer_age        34500 non-null  int64  
 16  customer_gender

In [9]:
print(df.shape)

(34500, 17)


In [10]:
print(df["returned"].unique())

<StringArray>
['No', 'Yes']
Length: 2, dtype: str


In [12]:
df["order_date"] = pd.to_datetime(df["order_date"],"%Y-%m-%d")

In [13]:
print(df.isnull().sum())

order_id              0
customer_id           0
product_id            0
category              0
price                 0
discount              0
quantity              0
payment_method        0
order_date            0
delivery_time_days    0
region                0
returned              0
total_amount          0
shipping_cost         0
profit_margin         0
customer_age          0
customer_gender       0
dtype: int64


In [14]:
df_clean = df[df["returned"] == "No"].copy()

In [15]:
print(df_clean)

      order_id customer_id product_id     category   price  discount  \
0      O100000      C17270    P234890         Home  164.08      0.15   
1      O100001      C17603    P228204      Grocery   24.73      0.00   
2      O100002      C10860    P213892  Electronics  175.58      0.05   
3      O100003      C15390    P208689  Electronics   63.67      0.00   
4      O100004      C15226    P228063         Home   16.33      0.15   
...        ...         ...        ...          ...     ...       ...   
34494  O134494      C14730    P245328       Sports   36.22      0.05   
34495  O134495      C13553    P231412      Fashion   10.44      0.20   
34497  O134497      C12986    P204345  Electronics  323.64      0.00   
34498  O134498      C17196    P217644         Home   23.09      0.05   
34499  O134499      C13330    P207047         Home   76.84      0.00   

       quantity payment_method order_date  delivery_time_days region returned  \
0             1    Credit Card 2023-12-23             

In [16]:
print("Original rows:", len(df))
print("After removing returns:", len(df_clean))

Original rows: 34500
After removing returns: 32597


In [18]:
snapshot_date = df_clean["order_date"].max() + pd.Timedelta(days=1)

In [20]:
rfm = df_clean.groupby("customer_id").agg(
    Recency = ("order_date", lambda x: (snapshot_date - x.max()).days),
    Frequency = ("order_id", "nunique"),
    Monetary = ("total_amount", sum)
).reset_index()

print(rfm.shape)
print(rfm.head(10))
print(rfm.describe())

(7877, 4)
  customer_id  Recency  Frequency  Monetary
0      C10000        5          2    210.58
1      C10001      194          5   3246.02
2      C10002      401          5    216.85
3      C10003       72          3    154.30
4      C10004      600          3    716.99
5      C10005      210          3    216.26
6      C10006      201          2    355.74
7      C10007       70          6    734.63
8      C10008       34          2    276.66
9      C10009      283          5    659.55
           Recency    Frequency      Monetary
count  7877.000000  7877.000000   7877.000000
mean    166.319030     4.138251    695.256707
std     151.922438     1.980365    800.426594
min       1.000000     1.000000      1.240000
25%      50.000000     3.000000    189.440000
50%     121.000000     4.000000    450.970000
75%     237.000000     5.000000    903.800000
max     729.000000    12.000000  13885.100000


In [22]:
rfm["R_score"] = pd.qcut(rfm["Recency"], 4, labels=[4,3,2,1])
rfm["F_score"] = pd.qcut(rfm["Frequency"].rank(method="first"),4, labels=[1,2,3,4,])
rfm["M_score"] = pd.qcut(rfm["Monetary"], 4, labels=[1,2,3,4])

rfm["RFM_Score"] = rfm["R_score"].astype(str) + rfm["F_score"].astype(str) + rfm["M_score"].astype(str)
print(rfm.head(10))
print(rfm["RFM_Score"].value_counts().head(10))

  customer_id  Recency  Frequency  Monetary R_score F_score M_score RFM_Score
0      C10000        5          2    210.58       4       1       2       412
1      C10001      194          5   3246.02       2       3       4       234
2      C10002      401          5    216.85       1       3       2       132
3      C10003       72          3    154.30       3       1       1       311
4      C10004      600          3    716.99       1       1       3       113
5      C10005      210          3    216.26       2       1       2       212
6      C10006      201          2    355.74       2       1       2       212
7      C10007       70          6    734.63       3       4       3       343
8      C10008       34          2    276.66       4       1       2       412
9      C10009      283          5    659.55       1       3       3       133
RFM_Score
111    624
444    400
344    302
443    248
211    240
244    207
112    190
222    188
333    186
433    184
Name: count, dtype: in

In [34]:
def segment_customer(row):
    r, f, m = int(row['R_score']), int(row['F_score']), int(row['M_score'])
    
    if r >= 3 and f >= 3 and m >= 3:
        return 'Champions'
    elif r >= 3 and f >= 2:
        return 'Loyal Customers'
    elif r >= 3 and f <= 2:
        return 'New/Promising'
    elif r == 2 and f >= 3:
        return 'Needs Attention'
    elif r <= 2 and f >= 3 and m >= 3:
        return 'At Risk'
    elif r <= 2 and f <= 2 and m >= 3:
        return 'Cant Lose Them'
    elif r <= 2 and f <= 2 and m <= 2:
        return 'Lost'
    else:
        return 'Others'

rfm['Segment'] = rfm.apply(segment_customer, axis=1)

print(rfm['Segment'].value_counts())
print(rfm.groupby('Segment')['Monetary'].sum().sort_values(ascending=False))

Segment
Champions          1826
Lost               1799
Loyal Customers    1582
Needs Attention     962
Cant Lose Them      727
New/Promising       536
At Risk             276
Others              169
Name: count, dtype: int64
Segment
Champions          2352793.78
Needs Attention     885148.58
Cant Lose Them      738879.98
Loyal Customers     682824.14
At Risk             322751.19
Lost                289762.08
New/Promising       158425.04
Others               45952.29
Name: Monetary, dtype: float64


In [32]:
rfm.columns

Index(['customer_id', 'Recency', 'Frequency', 'Monetary', 'R_score', 'F_score',
       'M_score', 'RFM_Score'],
      dtype='str')

In [35]:
# Export the full RFM table with segments to CSV
rfm.to_csv('rfm_analysis.csv', index=False)

print("Exported", len(rfm), "customer records to rfm_analysis.csv")

Exported 7877 customer records to rfm_analysis.csv


In [36]:
import os
print(os.path.abspath('rfm_analysis.csv'))

c:\Users\Admin\AppData\Local\Programs\Microsoft VS Code\rfm_analysis.csv
